## MINI-PROJEKT 3 - klasyfikator obrazów

* Wczytanie danych treningowych 
* Augmentacja danych
* Custom Dataset dla zbioru testowego - tenor + nazwa pliku
* Generowanie pliku preds.csv, zapis wszystkiego do zipa i wyslanie



* Budowa architektury CNN
* Pętla ucząca 
* Tuning hiperparametrów

In [27]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

### 1. Wczytanie danych treningowych & augmentacja

In [28]:
IMG_SIZE = 128 
BATCH_SIZE = 32

# transformacja dla treningowego (z augmentacja)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5), # losowe odbicie lustrzane
    transforms.RandomRotation(degrees=15),  # losowy obrot o max 15 stopni
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # zmiana kolorkow o max 20%
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalizacja
])


try:
    trainset = ImageFolder(root="train/", transform=train_transform)
    trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    print(f"Loaded train dataset: {len(trainset)} images, {len(trainset.classes)} classes.")
except FileNotFoundError:
    print("Couldnt fing 'train/' directory!!!!")

Loaded train dataset: 88011 images, 50 classes.


### 2. Wczytanie danych treningowych

In [29]:
# transformacja dla testowego (bez augmentacji)
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class CustomTestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if os.path.isfile(os.path.join(root_dir, f))]


    def __len__(self):
        return len(self.image_files)


    # zwracamy obraz i nazwe
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.root_dir, img_name)
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, img_name



try:
    testset = CustomTestDataset(root_dir="test/", transform=test_transform)
    testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"Loaded test dataset: {len(testset)} images.")
except FileNotFoundError:
    print("Couldnt fing 'test/' directory!!!!")

Loaded test dataset: 10000 images.


### Generowanie preds.csv

In [30]:
def generate_predictions(model, dataloader, device, output_filename="pred.csv"):

    model.eval() 
    predictions_list = []
    
    print(f"Generating preds to {output_filename}...")
    
    with torch.no_grad(): 
        for images, filenames in dataloader:
            images = images.to(device)
            
            outputs = model(images)


            _, predicted_classes = torch.max(outputs, 1) # pobranie indeksu klasy z najwyzszym p-nstwem
            predicted_classes = predicted_classes.cpu().numpy() # wyniki -> cpu 
            

            for filename, pred in zip(filenames, predicted_classes):
                predictions_list.append([filename, pred])
                
                
    df = pd.DataFrame(predictions_list)
    df.to_csv(output_filename, index=False, header=False)
    print(f"Done!! Saved {len(predictions_list)} predictions.")

### Tyci teścik zapisu

In [31]:
import torch.nn as nn

class DummyModel(nn.Module):
        def forward(self, x):
            return torch.randn(x.size(0), 50) # losowe tensory

if __name__ == "__main__":        
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dummy_model = DummyModel().to(device)
    
    generate_predictions(dummy_model, testloader, device, "pred.csv")

Generating preds to pred.csv...
Done!! Saved 10000 predictions.
